# Results walkthrough (supplementary)
The implementation lives in `src/readmission` and `scripts/`. This notebook only **reads** what the
scripts produced (`reports/tables`, `reports/figures`, `models/final_model.joblib`), so the reader can browse the results.
Run scripts 01 to 06 first.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pathlib import Path
T, F = Path("../reports/tables"), Path("../reports/figures")
pd.set_option("display.precision", 4)

## 1. Cohort and target

In [ ]:
display(pd.read_csv(T / "eda_cohort.csv"))
display(Image(F / "eda_target.png")); display(Image(F / "eda_signal.png"))

## 2. Untuned comparisons: model progression, imbalance remedies, feature ablations

In [ ]:
exp = pd.read_csv(T / "experiments_untuned.csv")
exp[["block", "config", "cv_ap_mean", "cv_ap_std", "cv_auc_mean", "brier", "precision_at_20pct", "recall_at_20pct"]]

## 3. Optuna-tuned models (same folds, same metrics)

In [ ]:
display(pd.read_csv(T / "tuned_models_cv.csv", index_col=0))
for m in ["logreg", "random_forest", "lightgbm", "catboost"]:
    p = F / f"optuna_history_{m}.png"
    if p.exists(): display(Image(p))

## 4. Calibration, threshold and the single test evaluation

In [ ]:
display(pd.read_csv(T / "calibration_methods_oof.csv", index_col=0))
display(Image(F / "threshold_cost_curve.png"))
display(pd.read_csv(T / "final_test_metrics.csv", index_col=0))
display(Image(F / "calibration_test.png")); display(Image(F / "final_test_curves.png"))

## 5. Explainability (SHAP)

In [ ]:
display(Image(F / "shap_global_bar.png")); display(Image(F / "shap_beeswarm.png"))
display(pd.read_csv(T / "shap_local_cases.csv"))

## 6. Fairness audit (race, equal opportunity)

In [ ]:
display(pd.read_csv(T / "fairness_summary.csv", index_col=0).T)
display(Image(F / "fairness_race.png"))

## 7. Scoring a new encounter with the frozen bundle (same code path as the API)

In [ ]:
import sys; sys.path.insert(0, "..")
from api.app import Encounter
from readmission.predict import score
rows = pd.DataFrame([Encounter().model_dump(),
                     Encounter(number_inpatient=4, n_prior_encounters=5, number_emergency=2).model_dump()])
score(rows)